In [ ]:
class MeanPooling(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, seq_embs, seq_mask, query=None):
        # seq_embs: (B, L, D), seq_mask: (B, L)
        mask = seq_mask.unsqueeze(-1).float()              # (B, L, 1)
        masked = seq_embs * mask
        denom = mask.sum(dim=1).clamp_min(self.eps)        # (B, 1)
        pooled = masked.sum(dim=1) / denom                 # (B, D)

        # 형식 맞추기용 dummy attention
        attn = seq_mask.float()
        attn = attn / attn.sum(dim=1, keepdim=True).clamp_min(self.eps)
        return pooled, attn

In [ ]:
class TopKPooling(nn.Module):
    def __init__(self, k=3, eps=1e-8):
        super().__init__()
        self.k = k
        self.eps = eps

    def forward(self, seq_embs, seq_mask, query=None):
        # norm 기준 상위 k개 선택 (query-agnostic)
        norms = torch.norm(seq_embs, dim=-1)                       # (B, L)
        norms = norms.masked_fill(~seq_mask, -1e9)

        k_eff = min(self.k, seq_embs.size(1))
        topk_vals, topk_idx = torch.topk(norms, k=k_eff, dim=1)   # (B, k)

        gathered = torch.gather(
            seq_embs,
            dim=1,
            index=topk_idx.unsqueeze(-1).expand(-1, -1, seq_embs.size(-1))
        )                                                          # (B, k, D)

        pooled = gathered.mean(dim=1)                              # (B, D)

        attn = torch.zeros_like(norms)
        attn.scatter_(1, topk_idx, 1.0 / k_eff)
        return pooled, attn

In [ ]:
class TopKSimPooling(nn.Module):
    def __init__(self, k=3, eps=1e-8):
        super().__init__()
        self.k = k
        self.eps = eps

    def forward(self, seq_embs, seq_mask, query):
        # query와 cosine similarity 높은 상위 k개 선택
        seq_norm = F.normalize(seq_embs, p=2, dim=-1)
        q_norm = F.normalize(query, p=2, dim=-1).unsqueeze(1)      # (B,1,D)

        sim = (seq_norm * q_norm).sum(dim=-1)                      # (B, L)
        sim = sim.masked_fill(~seq_mask, -1e9)

        k_eff = min(self.k, seq_embs.size(1))
        topk_vals, topk_idx = torch.topk(sim, k=k_eff, dim=1)

        gathered = torch.gather(
            seq_embs,
            dim=1,
            index=topk_idx.unsqueeze(-1).expand(-1, -1, seq_embs.size(-1))
        )                                                          # (B, k, D)

        pooled = gathered.mean(dim=1)

        attn = torch.zeros_like(sim)
        attn.scatter_(1, topk_idx, 1.0 / k_eff)
        return pooled, attn

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, tau=0.5, eps=1e-8):
        super().__init__()
        self.tau = tau
        self.eps = eps

    def forward(self, seq_embs, seq_mask, query):
        seq_norm = F.normalize(seq_embs, p=2, dim=-1)
        q_norm = F.normalize(query, p=2, dim=-1).unsqueeze(1)

        sim = (seq_norm * q_norm).sum(dim=-1) / self.tau           # (B, L)
        sim = sim.masked_fill(~seq_mask, -1e9)

        attn = F.softmax(sim, dim=-1)
        attn = attn * seq_mask.float()
        attn = attn / attn.sum(dim=1, keepdim=True).clamp_min(self.eps)

        pooled = torch.bmm(attn.unsqueeze(1), seq_embs).squeeze(1) # (B, D)
        return pooled, attn